# 🤖 Baseline Benchmark Modeling Notebook
**Semi-Automated Classification Workflow**

Supported use cases:
- Customer Churn Prediction
- Credit Risk Prediction
- Loan Default Prediction

> **Goal:** Prepare data, train multiple baseline models, compare performance, and identify the most important features.


## Section 1 — Load Libraries

First, install the libraries that are not pre-installed in Google Colab. Then import everything needed for modeling.

In [ ]:
# Install libraries not included in Google Colab by default
!pip install -q xgboost lightgbm catboost imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully.')

---
## Section 2 — Load Clean Dataset

Upload the `clean_dataset.csv` file produced by the Cleaning notebook.

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print(f'File loaded: {filename}')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Section 3 — Feature and Target Selection

Define which column is the target (what you want to predict) and which columns are features (inputs to the model).

> **Action required:** Set your target column name below.

In [ ]:
# ✏️ Set your target column name here
TARGET_COLUMN = 'Churn'  # <-- Change this to your actual target column

# -----------------------------------------------
if TARGET_COLUMN not in df.columns:
    raise ValueError(f'Column "{TARGET_COLUMN}" not found. Available: {list(df.columns)}')

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print(f'Target Column    : {TARGET_COLUMN}')
print(f'Number of Features : {X.shape[1]}')
print(f'Target Classes   : {sorted(y.unique())}')

---
## Section 4 — Encoding

Machine learning models require numerical inputs. This step converts categorical columns:
- **Binary categories** (Yes/No, True/False, Male/Female) → Label Encoding (0 or 1)
- **Multi-value categories** → One-Hot Encoding (new column per category)

In [ ]:
# Define binary value sets
BINARY_SETS = [
    {'yes', 'no'},
    {'true', 'false'},
    {'male', 'female'},
    {'0', '1'},
    {'y', 'n'}
]

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
binary_cols = []
multi_cols = []

for col in categorical_cols:
    unique_vals = set(X[col].dropna().str.lower().unique())
    if unique_vals in BINARY_SETS or len(unique_vals) == 2:
        binary_cols.append(col)
    else:
        multi_cols.append(col)

# Label Encoding for binary columns
le = LabelEncoder()
for col in binary_cols:
    X[col] = le.fit_transform(X[col].astype(str))

# One-Hot Encoding for multi-value columns
if multi_cols:
    X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

# Encode target if needed
if y.dtype == 'object':
    y = le.fit_transform(y.astype(str))
    y = pd.Series(y)

# Store feature names after encoding
feature_names = X.columns.tolist()
encoding_applied = f'Label Encoding: {len(binary_cols)} cols | One-Hot Encoding: {len(multi_cols)} cols'

print(f'Binary columns (Label Encoded)   : {binary_cols}')
print(f'Multi columns (One-Hot Encoded)  : {multi_cols}')
print(f'Total features after encoding    : {X.shape[1]}')

---
## Section 5 — Train Test Split

Split the dataset into 80% training and 20% testing. Stratification ensures both splits have the same class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Train Shape : {X_train.shape}')
print(f'Test Shape  : {X_test.shape}')

---
## Section 6 — Class Imbalance Check

Check if the target classes in the training set are balanced. A minority class below 30% is considered imbalanced.

In [ ]:
counts = y_train.value_counts()
percentages = y_train.value_counts(normalize=True) * 100

imbalance_df = pd.DataFrame({
    'Class': counts.index,
    'Count': counts.values,
    'Percentage (%)': percentages.values
})

print('--- Training Set Class Distribution ---')
display(imbalance_df.style.format({'Percentage (%)': '{:.2f}%'}))

min_pct = percentages.min()
IS_IMBALANCED = min_pct < 30

if IS_IMBALANCED:
    print(f'\n⚠️  Status: IMBALANCED  (minority class = {min_pct:.1f}%) — SMOTE will be applied.')
else:
    print(f'\n✅ Status: BALANCED  (minority class = {min_pct:.1f}%) — No balancing needed.')

---
## Section 7 — Balancing

If the dataset is imbalanced, **SMOTE** (Synthetic Minority Over-sampling Technique) is applied to the training set only. The test set is never modified.

In [ ]:
balancing_applied = 'No'

if IS_IMBALANCED:
    print('Applying SMOTE to training data...')
    print(f'Before SMOTE: {dict(y_train.value_counts())}')

    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    y_train = pd.Series(y_train)

    print(f'After SMOTE : {dict(y_train.value_counts())}')
    print(f'New Train Shape: {X_train.shape}')
    balancing_applied = 'Yes (SMOTE)'
else:
    print('Dataset is balanced. Skipping SMOTE.')

---
## Section 8 — Scaling

Some models (like Logistic Regression) perform better with scaled data. Tree-based models do not require scaling.

Two versions of the training and test data are created:
- **Scaled** — for Logistic Regression
- **Non-scaled** — for all tree-based models

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Non-scaled (keep as is)
X_train_raw = X_train.values
X_test_raw  = X_test.values

print('Scaled and non-scaled datasets are ready.')
print(f'Scaled Train Shape : {X_train_scaled.shape}')
print(f'Scaled Test Shape  : {X_test_scaled.shape}')

---
## Phase 2 — Model Discovery

## Section 9 — Baseline Models

Train 7 classification models using default parameters. No tuning is applied at this stage — this is a baseline comparison only.

In [ ]:
# Define models
# (model_name, model_object, use_scaled_data)
models = [
    ('Logistic Regression', LogisticRegression(random_state=42, max_iter=1000), True),
    ('Decision Tree',       DecisionTreeClassifier(random_state=42),            False),
    ('Random Forest',       RandomForestClassifier(random_state=42, n_jobs=-1), False),
    ('Extra Trees',         ExtraTreesClassifier(random_state=42, n_jobs=-1),   False),
    ('XGBoost',             XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0), False),
    ('LightGBM',            LGBMClassifier(random_state=42, verbose=-1),        False),
    ('CatBoost',            CatBoostClassifier(random_state=42, verbose=0),     False),
]

trained_models = {}

print('Training baseline models...')
print('-' * 35)

for name, model, use_scaled in models:
    Xtr = X_train_scaled if use_scaled else X_train_raw
    model.fit(Xtr, y_train)
    trained_models[name] = (model, use_scaled)
    print(f'  ✅ {name}')

print('-' * 35)
print('All models trained.')

---
## Section 10 — Model Evaluation

Evaluate each model on the test set using five metrics. Results are sorted by ROC AUC (highest first).

In [ ]:
results = []

for name, (model, use_scaled) in trained_models.items():
    Xte = X_test_scaled if use_scaled else X_test_raw

    y_pred = model.predict(Xte)

    # Use predict_proba for ROC AUC if available
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(Xte)[:, 1]
        roc = roc_auc_score(y_test, y_proba)
    else:
        roc = roc_auc_score(y_test, y_pred)

    results.append({
        'Model':     name,
        'ROC AUC':   round(roc, 4),
        'F1 Score':  round(f1_score(y_test, y_pred, average='weighted'), 4),
        'Precision': round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, average='weighted'), 4),
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
    })

results_df = pd.DataFrame(results).sort_values('ROC AUC', ascending=False).reset_index(drop=True)
results_df.index += 1  # Start rank from 1

print('--- Model Comparison (sorted by ROC AUC) ---')
display(results_df)

---
## Section 11 — Best Model Selection

The model with the highest ROC AUC is automatically selected as the best baseline model.

In [ ]:
best_row = results_df.iloc[0]
best_model_name = best_row['Model']
best_model_obj, best_use_scaled = trained_models[best_model_name]

print('=' * 42)
print('              BEST MODEL')
print('=' * 42)
print(f'  Model Name : {best_model_name}')
print(f'  ROC AUC    : {best_row["ROC AUC"]}')
print(f'  F1 Score   : {best_row["F1 Score"]}')
print(f'  Precision  : {best_row["Precision"]}')
print(f'  Recall     : {best_row["Recall"]}')
print(f'  Accuracy   : {best_row["Accuracy"]}')
print('=' * 42)

---
## Phase 3 — Model Interpretation

## Section 12 — Feature Importance Ranking

**Permutation Importance** measures how much model performance drops when a feature's values are randomly shuffled. A larger drop means the feature is more important.

This method works with all model types and is easy to understand.

In [ ]:
Xte_best = X_test_scaled if best_use_scaled else X_test_raw

print(f'Computing permutation importance for: {best_model_name}')
print('This may take a moment...')

perm = permutation_importance(
    best_model_obj, Xte_best, y_test,
    n_repeats=10,
    random_state=42,
    scoring='roc_auc'
)

importance_df = pd.DataFrame({
    'Feature':           feature_names,
    'Importance Score':  perm.importances_mean
})

importance_df = importance_df.sort_values('Importance Score', ascending=False).reset_index(drop=True)
importance_df.index += 1
importance_df['Importance Score'] = importance_df['Importance Score'].round(4)

top20 = importance_df.head(20)

print('\n--- Top 20 Feature Importance Ranking ---')
display(top20)

# Bar chart — top 20 only
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=top20, x='Importance Score', y='Feature', palette='Blues_r', ax=ax)
ax.set_title(f'Top 20 Feature Importance — {best_model_name}')
ax.set_xlabel('Importance Score (Permutation)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## Section 13 — Top Feature Recommendation

The top 10 most important features are recommended for a future compact model. Using fewer features makes deployment simpler and reduces the number of inputs required from users.

In [ ]:
top10 = importance_df.head(10).reset_index()
top10.columns = ['Rank', 'Feature', 'Importance Score']

top10_features = top10['Feature'].tolist()

print('=' * 48)
print('        TOP FEATURE RECOMMENDATION')
print('=' * 48)
display(top10.style.format({'Importance Score': '{:.4f}'}))
print('=' * 48)
print('Purpose:')
print('  - Simpler deployment')
print('  - Fewer user inputs required')
print('  - Candidate for compact model')

---
## Section 14 — Benchmark Summary

A complete summary of the entire baseline benchmark workflow.

In [ ]:
print('=' * 52)
print('        BASELINE BENCHMARK SUMMARY')
print('=' * 52)
print(f'  Dataset Shape        : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'  Total Features       : {len(feature_names)}')
print(f'  Encoding Applied     : {encoding_applied}')
print(f'  Balancing Applied    : {balancing_applied}')
print(f'  Models Tested        : {len(models)}')
print(f'  Best Model           : {best_model_name}')
print(f'  Best ROC AUC         : {best_row["ROC AUC"]}')
print(f'')
print(f'  Top 10 Features:')
for i, feat in enumerate(top10_features, 1):
    print(f'    {i:>2}. {feat}')
print('=' * 52)
print()
print('  ✅ Next Step: Run Best Model Tuning Notebook')
print('=' * 52)